## Planning Pattern

Notes from Andrew Ng's Agentic AI course — third pattern after reflection and tool use.

So the idea here is pretty different from the previous two. Reflection was about looping to improve quality, tool use was about letting the model call external APIs. Planning is about getting the model to think through *what it needs to do* before it actually does anything.

Instead of the model just reacting to a query and picking the next action on the fly, you first ask it: "given this task, what's your step-by-step plan?" Then you execute that plan one step at a time, feeding each step's output into the next one as context.

I think the reason this works well is that planning forces the model to reason about the whole problem upfront rather than being shortsighted. A reactive agent that just picks the next best action can paint itself into corners. A planner sees the dependencies between steps before committing to anything.

### How it actually works

There are two phases — planning and execution — and they're kept separate on purpose.

```
User query → LLM (planning phase) → step-by-step plan
                                          ↓
                             Step 1 text → LLM → tool call → result
                                          ↓
                     Step 1 output + Step 2 text → LLM → tool call → result
                                          ↓
                     Step 2 output + Step 3 text → LLM → tool call → result
                                          ↓
                                      Final answer
```

The planning call just produces a list of steps — it doesn't execute anything. Then you loop through the steps, run each one, and pass the output forward. Each step is its own LLM call with the previous results in context.

The separation between planning and execution is actually really useful. It means you can *look at the plan* before you run it. For anything where a wrong action is hard to undo (sending an email, deleting data, deploying something), you can stick a human review step in between. If the plan looks off, you catch it before anything happens.

### Three formats a plan can take

The course goes through three ways the planner can output a plan. They're not just stylistic choices — each one has real implications for how you execute it.

**Text** — just natural language steps:
1. Use `get_item_descriptions` to find round sunglasses
2. Use `check_inventory` to check stock
3. Use `get_item_price` to filter under $100

Honestly this one is mostly useful for display or when you want a human to read the plan. Actually executing it programmatically is a pain — you have to parse natural language to figure out which tool to call, which always breaks on edge cases.

**JSON** — structured, with explicit tool names and args per step:
```json
{
  "plan": [
    {"step": 1, "description": "Find round sunglasses", "tool": "get_item_descriptions", "args": {"query": "round sunglasses"}},
    {"step": 2, "description": "Check stock", "tool": "check_inventory", "args": {"items": "results from step 1"}},
    {"step": 3, "description": "Filter by price", "tool": "get_item_price", "args": {"items": "results from step 2"}}
  ]
}
```

This is much easier to work with in code. Tool names are explicit, you can validate the plan before running it, catch unknown tools, etc. The catch is that you're stuck with whatever tools you've predefined — the plan can only call what you've already built.

**Code** — the model writes Python that gets executed directly. This is the one that actually surprised me, and it's where we'll spend most of the notebook.

### Why code execution beats predefined tools for data tasks

This was probably the most useful insight in this section of the course. The example they used: *"Which month had the highest average sales of hot chocolate?"*

If you're using JSON planning with predefined tools (`filter_rows`, `get_column_mean`, `sum_rows`, etc.), your plan ends up looking like:
1. Filter rows where coffee_name is "Hot Chocolate" for January, get mean
2. Filter rows where coffee_name is "Hot Chocolate" for February, get mean
3. ... repeat for every month ...
4. Compare all 12 averages, return the winner

That's a lot of steps and a lot of LLM calls, and the only reason it's so verbose is that your `filter_rows` tool doesn't understand "group by month". You'd have to build a new tool for that specific operation.

With code it's just:
```python
import pandas as pd
df = pd.read_csv('coffee_sales.csv')
df['date'] = pd.to_datetime(df['date'])
hot_choc = df[df['coffee_name'] == 'Hot Chocolate']
monthly_avg = hot_choc.groupby(df['date'].dt.month)['price'].mean()
print(monthly_avg.idxmax())
```

Done. Pandas already has `groupby`. You didn't need to build anything. This generalizes — whenever the task is data manipulation or analysis, code gives you access to the entire NumPy/Pandas ecosystem for free instead of needing a custom tool for every operation.

The course showed benchmark numbers across models (GPT-4, Claude-2, GPT-3.5, Gemini, LLaMA): **Code as Action beats JSON as Action beats Text as Action** on successful task completion, consistently. The gap is biggest on stronger models. Makes sense — stronger models write better code.

In [ ]:
import io
import json
import re
import textwrap
import contextlib
from typing import Any, Dict, List, Optional

from openai import OpenAI

client = OpenAI()

print("Setup complete.")

### Example 1: Customer service agent with JSON planning

First example from the course slides. Customer asks: *"Do you have any round sunglasses in stock that are under $100?"*

Three tools available: `get_item_descriptions`, `check_inventory`, `get_item_price`. The interesting part is that the model needs to figure out the right order AND knows that step 2 depends on step 1's output and step 3 depends on step 2's. It handles the data dependency automatically.

Let's see what plan it generates and then run it.

In [ ]:
# Mock tool implementations — in production these would hit real APIs or databases

def get_item_descriptions(query: str) -> Dict[str, Any]:
    """Search product catalog by description."""
    results = [
        {"id": "SG-001", "name": "Classic Round Sunglasses", "style": "round"},
        {"id": "SG-002", "name": "Vintage Round Frames", "style": "round"},
        {"id": "SG-003", "name": "Oversized Round Shades", "style": "round"},
    ]
    return {"items": results, "query": query}


def check_inventory(item_ids: List[str]) -> Dict[str, Any]:
    """Check which items are currently in stock."""
    stock = {"SG-001": True, "SG-002": False, "SG-003": True}
    return {
        "in_stock": [i for i in item_ids if stock.get(i, False)],
        "out_of_stock": [i for i in item_ids if not stock.get(i, False)],
    }


def get_item_price(item_ids: List[str]) -> Dict[str, Any]:
    """Look up current prices for items."""
    prices = {"SG-001": 89.99, "SG-002": 120.00, "SG-003": 74.99}
    return {
        "prices": [
            {"id": item_id, "price": prices.get(item_id, 0)} for item_id in item_ids
        ]
    }


CUSTOMER_SERVICE_TOOLS = {
    "get_item_descriptions": get_item_descriptions,
    "check_inventory": check_inventory,
    "get_item_price": get_item_price,
}

print("Customer service tools loaded:", list(CUSTOMER_SERVICE_TOOLS.keys()))

In [ ]:
PLANNER_SYSTEM_PROMPT = """
You are a planning agent. You have access to the following tools:

- get_item_descriptions(query: str) — Search the product catalog by description. Returns a list of matching items with their IDs.
- check_inventory(item_ids: list) — Check which item IDs are currently in stock.
- get_item_price(item_ids: list) — Get current prices for a list of item IDs.

Given a user request, create a step-by-step plan in JSON format to fulfill it.
Each step should have: step (number), description (what this step does), tool (tool name), and args (arguments as a dict).
For args that depend on a previous step's output, use the string "results from step N".

Return ONLY valid JSON. No explanation outside the JSON.
""".strip()


def create_plan(user_query: str) -> List[Dict[str, Any]]:
    """Ask the LLM to produce a JSON plan for the given query."""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
            {"role": "user", "content": user_query},
        ],
        response_format={"type": "json_object"},
    )
    plan = json.loads(response.choices[0].message.content)
    return plan.get("plan", [])


user_query = "Do you have any round sunglasses in stock that are under $100?"
plan = create_plan(user_query)

print("Generated plan:")
print(json.dumps(plan, indent=2))

In [ ]:
def execute_plan(plan: List[Dict[str, Any]], tools: Dict, user_query: str) -> str:
    """
    Execute a JSON plan step by step.

    Each step's output is stored and injected into the next step's context
    via the LLM — so the model can use step 1's results when setting up step 2's args.
    """
    step_outputs = {}
    conversation = [{"role": "user", "content": user_query}]

    for step in plan:
        step_num = step["step"]
        tool_name = step["tool"]
        description = step["description"]

        print(f"\nStep {step_num}: {description}")
        print(f"  Tool: {tool_name}")

        # Build context for this step, including all previous outputs
        step_context = f"Execute step {step_num}: {description}\n"
        step_context += f"Tool to call: {tool_name}\n"
        step_context += f"Planned args: {json.dumps(step['args'])}\n"

        if step_outputs:
            step_context += "\nPrevious step outputs:\n"
            for prev_step, prev_output in step_outputs.items():
                step_context += f"Step {prev_step}: {json.dumps(prev_output)}\n"

        step_context += "\nDetermine the actual arguments to pass to the tool given the above context. Return ONLY a JSON object with the args."

        # Ask the LLM to resolve argument placeholders against actual previous outputs
        conversation.append({"role": "user", "content": step_context})
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=conversation,
            response_format={"type": "json_object"},
        )

        resolved_args = json.loads(response.choices[0].message.content)
        print(f"  Resolved args: {resolved_args}")

        result = tools[tool_name](**resolved_args) if tool_name in tools else {"error": f"Unknown tool: {tool_name}"}
        print(f"  Result: {json.dumps(result)}")

        step_outputs[step_num] = result
        conversation.append({"role": "assistant", "content": response.choices[0].message.content})
        conversation.append({"role": "user", "content": f"Step {step_num} result: {json.dumps(result)}"})

    # Final synthesis — produce an answer from all collected step outputs
    conversation.append({
        "role": "user",
        "content": (
            f"All steps are complete. Here are the results:\n{json.dumps(step_outputs, indent=2)}\n\n"
            f"Please answer the original question: '{user_query}'"
        )
    })

    final = client.chat.completions.create(model="gpt-4o", messages=conversation)
    return final.choices[0].message.content


answer = execute_plan(plan, CUSTOMER_SERVICE_TOOLS, user_query)
print(f"\nFinal answer:\n{answer}")

### Example 2: Email assistant

Second example from the slides. User: *"Reply to that email invitation from Bob about dinner in New York and tell him I'll attend. Then archive his email."*

What I find interesting about this one compared to the sunglasses example: the user is assuming the agent already knows which email they mean ("that email invitation"). So step 1 has to be a search — you can't reply without knowing the email ID first.

The dependency chain here is: search → get ID → reply using ID + archive using same ID. The planner needs to reason that both send and archive depend on the search result, not just send. Worth watching whether the generated plan picks that up correctly.

In [ ]:
def search_email(from_sender: Optional[str] = None, keywords: Optional[List[str]] = None) -> Dict[str, Any]:
    """Search inbox for emails matching sender or keywords."""
    mock_emails = [
        {
            "id": "email_001",
            "from": "bob@example.com",
            "subject": "Dinner in New York?",
            "body": "Hey! Are you free for dinner in New York next Thursday? Would love to catch up.",
            "date": "2026-04-10",
        }
    ]
    return {"emails": mock_emails, "count": len(mock_emails)}


def send_email(to: str, subject: str, body: str, reply_to_id: Optional[str] = None) -> Dict[str, Any]:
    """Send an email."""
    print(f"  [MOCK] Sending to {to}: '{subject}'")
    return {"success": True, "sent_to": to, "subject": subject}


def move_email(email_id: str, destination_folder: str) -> Dict[str, Any]:
    """Move an email to a folder."""
    print(f"  [MOCK] Moving {email_id} to '{destination_folder}'")
    return {"success": True, "email_id": email_id, "moved_to": destination_folder}


def delete_email(email_id: str) -> Dict[str, Any]:
    """Delete an email."""
    print(f"  [MOCK] Deleting {email_id}")
    return {"success": True, "email_id": email_id}


EMAIL_TOOLS = {
    "search_email": search_email,
    "send_email": send_email,
    "move_email": move_email,
    "delete_email": delete_email,
}

EMAIL_PLANNER_PROMPT = """
You are a planning agent with access to these email tools:

- search_email(from_sender: str, keywords: list) — Search inbox. Returns email IDs and content.
- send_email(to: str, subject: str, body: str, reply_to_id: str) — Send or reply to an email.
- move_email(email_id: str, destination_folder: str) — Move an email to a folder (e.g. 'archive').
- delete_email(email_id: str) — Permanently delete an email.

Given a user request, create a step-by-step JSON plan. Return ONLY valid JSON with a 'plan' array.
Each step: step, description, tool, args. Use "results from step N" as a placeholder when args depend on a prior step.
""".strip()


email_query = "Reply to that email invitation from Bob about dinner in New York and tell him I'll attend. Then archive his email."

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": EMAIL_PLANNER_PROMPT},
        {"role": "user", "content": email_query},
    ],
    response_format={"type": "json_object"},
)

email_plan = json.loads(response.choices[0].message.content).get("plan", [])
print("Email assistant plan:")
print(json.dumps(email_plan, indent=2))

In [ ]:
email_answer = execute_plan(email_plan, EMAIL_TOOLS, email_query)
print(f"\nFinal answer:\n{email_answer}")

### Example 3: Planning with code execution

This is the variant Andrew Ng spent the most time on, and honestly it's the most interesting one. Instead of picking from a fixed set of tools, the model writes Python code that actually runs.

The pattern is simple: system prompt tells the model to wrap any code it wants to run inside `<execute_python>` tags. Your application extracts that code, runs it, captures stdout, and feeds it back into the conversation. The model reads the output and either writes more code or gives a final answer.

It's basically the same loop as tool use — model outputs something structured, you execute it, result goes back in as context — except the "tool" is a code interpreter and the model can write whatever code it wants. No need to have anticipated every possible operation in advance.

One thing to be aware of: the `exec()` approach used here is fine for local experimentation. In production you'd want a proper sandbox with resource limits and no unrestricted filesystem/network access.

In [ ]:
def extract_python_code(text: str) -> Optional[str]:
    """Pull code out of <execute_python>...</execute_python> tags."""
    match = re.search(r"<execute_python>(.*?)</execute_python>", text, re.DOTALL)
    return match.group(1).strip() if match else None


def execute_python(code: str) -> str:
    """
    Run Python code and capture stdout.

    Uses exec() with a shared namespace — for production you'd want a proper
    sandbox (restricted builtins, resource limits, timeout). Fine for demos.
    """
    stdout_capture = io.StringIO()
    namespace: Dict[str, Any] = {}
    try:
        with contextlib.redirect_stdout(stdout_capture):
            exec(code, namespace)  # noqa: S102
        output = stdout_capture.getvalue()
        return output if output else "(no output)"
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"


# Quick sanity check
test_code = """
x = [1, 2, 3, 4, 5]
print(f"Sum: {sum(x)}, Mean: {sum(x)/len(x)}")
"""
print("Code execution test:", execute_python(test_code))

In [ ]:
import pandas as pd
import numpy as np

# Create the coffee sales dataset from the course slides
np.random.seed(42)
dates = pd.date_range(start="2024-01-01", end="2025-03-31", freq="D")
coffee_names = ["Hot Chocolate", "Cappuccino", "Latte", "Espresso", "Americano"]
sizes = ["S", "M", "L"]

records = []
for date in dates:
    month = date.month
    # Hot chocolate sells more in winter — seasonality baked in
    weights = {
        "Hot Chocolate": 0.4 if month in [11, 12, 1, 2] else 0.1,
        "Cappuccino": 0.25, "Latte": 0.25, "Espresso": 0.1, "Americano": 0.05,
    }
    total = sum(weights.values())
    normalized = [w / total for w in weights.values()]

    for _ in range(np.random.randint(20, 60)):
        name = np.random.choice(coffee_names, p=normalized)
        size = np.random.choice(sizes)
        base = {"Hot Chocolate": 3.50, "Cappuccino": 4.00, "Latte": 4.50, "Espresso": 2.50, "Americano": 3.00}
        adj = {"S": -0.50, "M": 0.0, "L": 0.75}
        price = round(base[name] + adj[size] + np.random.uniform(-0.25, 0.25), 2)
        records.append({"date": date.strftime("%Y-%m-%d"), "price": price, "coffee_name": name, "size": size})

df = pd.DataFrame(records)
df.to_csv("/tmp/coffee_sales.csv", index=False)

print(f"Dataset: {len(df)} transactions")
print(df.head(8).to_string(index=False))

In [ ]:
CODE_EXECUTION_PROMPT = """
You are a data analysis assistant with access to a Python code interpreter.

To run code, wrap it in <execute_python> and </execute_python> tags:

<execute_python>
import pandas as pd
df = pd.read_csv('/tmp/coffee_sales.csv')
print(df.head())
</execute_python>

The stdout from your code will be shown to you so you can continue reasoning.
Use pandas for data manipulation. The data is at /tmp/coffee_sales.csv
with columns: date, price, coffee_name, size.
""".strip()


def code_execution_agent(user_query: str, max_iterations: int = 5) -> str:
    """
    Agent that plans and executes by writing Python code.

    Each iteration:
    1. LLM responds with code in <execute_python> tags
    2. We extract and run the code, capture stdout
    3. Feed the output back to the LLM as context
    4. Repeat until the LLM gives a final answer (no code tags)
    """
    messages = [
        {"role": "system", "content": CODE_EXECUTION_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for iteration in range(max_iterations):
        print(f"\n[iteration {iteration + 1}]")

        response = client.chat.completions.create(model="gpt-4o", messages=messages)
        assistant_message = response.choices[0].message.content
        messages.append({"role": "assistant", "content": assistant_message})

        code = extract_python_code(assistant_message)

        # No code tags means the model has a final answer — stop here
        if code is None:
            print("No code to execute — model has final answer.")
            return assistant_message

        print("Executing:")
        print(textwrap.indent(code, "  "))

        output = execute_python(code)
        print(f"Output: {output}")

        messages.append({
            "role": "user",
            "content": f"Code output:\n{output}\n\nContinue with your analysis."
        })

    return "Reached max iterations."


print("=" * 60)
print("QUERY: Which month had the highest average sales of hot chocolate?")
print("=" * 60)
result = code_execution_agent("Which month had the highest average sales of hot chocolate?")
print(f"\nFINAL ANSWER:\n{result}")

In [ ]:
# Another query that would require 5+ predefined tools but is trivial with code
print("=" * 60)
print("QUERY: What were the amounts of the last 5 transactions?")
print("=" * 60)
result2 = code_execution_agent("What were the amounts of the last 5 transactions?")
print(f"\nFINAL ANSWER:\n{result2}")

### Agentic software coders — the peak use case

Andrew Ng specifically called out highly agentic coding assistants as the best example of planning in the wild. Worth unpacking why.

When you ask one of these tools to build something non-trivial, it doesn't just start writing code. It first figures out a plan — something like: design the schema, build the core models, add the API layer, write tests, run tests, fix failures, wire it together. That's not a reactive "what's the next token" pattern — it's actual upfront reasoning about what order to do things in and what depends on what.

The really powerful part is the test-as-you-go loop. The agent writes a component, runs the tests, gets back an error, reads the error, and goes back to fix the earlier code. That's reflection + code execution + planning all running together. It can catch and fix architectural mistakes mid-way through instead of finishing everything and discovering it's broken at the end.

The connection to the code execution example above is that they're the same pattern at different scales. One asks "which month had the highest hot chocolate sales" and runs a few lines of pandas. The other builds a full API and runs a test suite. The loop is identical — write code, execute it, get feedback, decide what to do next.

In [ ]:
SOFTWARE_PLANNER_PROMPT = """
You are a senior software architect. Given a software requirement, produce a detailed implementation plan.

The plan should specify:
- Components to build, in the order they should be built
- What each component does and what it depends on
- Where tests should be written and run

Return your plan as JSON with a 'plan' array.
Each item: step, component, description, dependencies, test_after (bool).
""".strip()


def plan_software_task(requirement: str) -> List[Dict[str, Any]]:
    """Generate a structured implementation plan for a software task."""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SOFTWARE_PLANNER_PROMPT},
            {"role": "user", "content": requirement},
        ],
        response_format={"type": "json_object"},
    )
    result = json.loads(response.choices[0].message.content)
    return result.get("plan", [])


requirement = """
Build a simple REST API for a to-do list app. It should support creating tasks,
listing all tasks, marking tasks as complete, and deleting tasks.
Use FastAPI and SQLite. Include tests.
""".strip()

software_plan = plan_software_task(requirement)

print("Software implementation plan:")
for step in software_plan:
    print(f"\nStep {step['step']}: {step.get('component', '')}")
    print(f"  {step.get('description', '')}")
    print(f"  Depends on: {step.get('dependencies', 'none')}")
    print(f"  Run tests after: {step.get('test_after', False)}")

### Replanning — what happens when something goes wrong

The course doesn't go deep on this but it's worth thinking about. Plans fail. A tool returns nothing, inventory is empty, the data has unexpected nulls — the agent needs to handle it rather than just crashing or hallucinating a recovery.

The approach: if a step's output breaks your assumptions for the next step, call the planner again with the current state as context. Give it: what the original goal was, which steps completed successfully, what the failed step returned, and what tools are available. Let it generate a revised plan from that point.

It's a small addition but it's the difference between an agent that works only on the happy path and one that actually recovers from real-world messiness.

In [ ]:
REPLANNER_PROMPT = """
You are a planning agent. A step in your plan has encountered an issue or unexpected result.
Given the original goal, the steps completed so far, and what went wrong,
produce a revised plan for the remaining work.

Return JSON with a 'revised_plan' array. Each step: step, description, tool, args.
""".strip()


def replan(
    original_goal: str,
    completed_steps: List[Dict],
    issue: str,
    available_tools: List[str],
) -> List[Dict[str, Any]]:
    """
    Generate a revised plan when something goes wrong mid-execution.

    Passes the original goal, what's been done so far, and what failed
    so the planner can reason about what to do differently.
    """
    context = f"""
Original goal: {original_goal}

Completed steps:
{json.dumps(completed_steps, indent=2)}

Issue encountered: {issue}

Available tools: {', '.join(available_tools)}

Please produce a revised plan for the remaining work.
""".strip()

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": REPLANNER_PROMPT},
            {"role": "user", "content": context},
        ],
        response_format={"type": "json_object"},
    )
    result = json.loads(response.choices[0].message.content)
    return result.get("revised_plan", [])


# Simulate: check_inventory returned empty — all items out of stock
completed = [
    {
        "step": 1,
        "tool": "get_item_descriptions",
        "result": {"items": [{"id": "SG-001", "name": "Classic Round Sunglasses"}]},
    },
    {
        "step": 2,
        "tool": "check_inventory",
        "result": {"in_stock": [], "out_of_stock": ["SG-001"]},
    },
]

revised = replan(
    original_goal="Find round sunglasses in stock under $100",
    completed_steps=completed,
    issue="All matching items are out of stock. No items to check prices for.",
    available_tools=["get_item_descriptions", "check_inventory", "get_item_price", "notify_user"],
)

print("Revised plan after inventory failure:")
print(json.dumps(revised, indent=2))

### How this connects to the other patterns

Planning doesn't really stand alone — it tends to pull in the other patterns as you make it more capable.

**With tool use** — JSON planning is basically structured tool use with ordering added. The planner decides which tools to call and in what sequence; the execution loop from the tool use pattern runs them. Most planning agents are just tool-use agents with an explicit planning phase up front.

**With reflection** — you can add a check after each step: did this output make sense? Does it match what the plan expected? If not, replan. Adds latency but catches problems before they compound across multiple steps.

**With code execution** — the code IS the plan. Each iteration the model writes the next piece of code based on what the previous piece returned. Dynamic and adaptive rather than fixed upfront.

**All four together** — that's basically what agentic software coders do. Plan the architecture, write code, run tests (reflection via external feedback), use file system and terminal tools. The four patterns aren't alternatives to each other — they stack.

### Things that tripped me up / things to watch for

**The plan can look totally reasonable and still be wrong.** The model generates confident-looking plans regardless of whether the ordering makes sense. Always inspect the JSON before running it, especially if there are irreversible steps. Validate that tool names exist, that step dependencies are correct, that nothing is trying to use "results from step 3" before step 3 runs.

**Argument placeholders are not real arguments.** When a step says `"args": {"items": "results from step 1"}`, that string is just a human-readable note — it's not an actual value. Something in your code has to resolve it against the real step 1 output before calling the tool. The `execute_plan` implementation above handles this with an extra LLM call per step, which works but costs tokens. For simple cases you can do it deterministically with string substitution.

**`exec()` is not a sandbox.** Fine here, but if you're building anything user-facing, the model can write code that reads arbitrary files, makes network requests, or eats memory. Use Docker, WASM, or a cloud code interpreter for real isolation.

**Long plans lose the plot.** By step 8 or 9, the model sometimes forgets what step 3 returned. Keeping all step outputs in the message history helps but blows up the context window fast. Worth summarizing intermediate results if your plans go beyond 5-6 steps.

**Don't use planning for simple tasks.** A single tool call doesn't need a planning phase. The overhead (an extra LLM call, more tokens, more latency) only pays off when the task genuinely has multiple steps with ordering constraints that aren't obvious upfront.

### Quick reference

```
Three plan formats:

  Text       →  readable, hard to execute programmatically
  JSON       →  structured, limited to predefined tools
  Code       →  most flexible, best performance on data tasks

Performance ranking (from course benchmark, all models tested):
  Code as Action  >  JSON as Action  >  Text as Action

When planning is worth it:
  - Multiple steps with ordering dependencies
  - You need to inspect/approve before running
  - Reactive step-by-step keeps going off track

When it's not worth it:
  - Single tool calls
  - Simple lookups
  - Anything where the overhead is bigger than the benefit

Best real-world application: agentic software coders
  plan → code → test → fix → repeat
  (all four patterns running together)
```